In [3]:
# ══════════════════════════════════════════
# BLOC 1 : IMPORTS ET CONFIGURATION DU RENDU
# ══════════════════════════════════════════

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import time
import warnings

# river : librairie online learning
from river import tree, neighbors, linear_model, metrics

# Configuration du rendu pour VS Code (évite les graphiques vides)
pio.renderers.default = 'vscode'
warnings.filterwarnings('ignore')

# Dictionnaire global des couleurs pour la cohérence
couleurs = {
    'HoeffdingTree': '#378ADD',
    'KNN'          : '#1D9E75',
    'SGD'          : '#7F77DD',
}

print('✓ Bloc 1 : Imports et Configuration OK')

✓ Bloc 1 : Imports et Configuration OK


In [4]:
# ══════════════════════════════════════════
# CHARGEMENT
# ══════════════════════════════════════════
df = pd.read_csv('../data/unsw-nb15/UNSW_NB15_training-set.csv')
print(f'Dataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

# ══════════════════════════════════════════
# PRÉTRAITEMENT EN 4 ÉTAPES
# ══════════════════════════════════════════

# 1. Supprimer colonnes inutiles
df_clean = df.drop(columns=['id', 'attack_cat'])

# 2. Encoder texte → nombres
#    tcp=0, udp=1, icmp=2 ... (river n'accepte que des nombres)
for col in ['proto', 'service', 'state']:
    df_clean[col] = pd.Categorical(df_clean[col]).codes
    print(f'  {col} encodé en nombres')

# 3. Séparer features et label
X = df_clean.drop(columns=['label'])
y = df_clean['label']

# 4. Normalisation min-max → tout entre [0, 1]
#    Formule : (valeur - min) / (max - min)
for col in X.columns:
    mn, mx = X[col].min(), X[col].max()
    if mx - mn > 0:
        X[col] = (X[col] - mn) / (mx - mn)

print(f'\n✓ Prétraitement terminé')
print(f'  X : {X.shape[0]:,} lignes × {X.shape[1]} features')
print(f'  y : {(y==0).sum():,} Normal | {(y==1).sum():,} Attaque')

Dataset : 82,332 lignes × 45 colonnes
  proto encodé en nombres
  service encodé en nombres
  state encodé en nombres

✓ Prétraitement terminé
  X : 82,332 lignes × 42 features
  y : 37,000 Normal | 45,332 Attaque


In [5]:
# ══════════════════════════════════════════
# SIMULATEUR DE FLUX
# Lit le CSV ligne par ligne → simule un réseau en direct
# ══════════════════════════════════════════

def simuler_flux(X, y, limite=None):
    """
    Génère les données une par une comme un vrai flux réseau.
    
    Args:
        X     : DataFrame des features
        y     : Série des labels
        limite: nombre max d'exemples (None = tout)
    
    Yields:
        (x_dict, label) → un dictionnaire + le vrai label
    """
    n = len(X) if limite is None else min(limite, len(X))
    for i in range(n):
        x     = X.iloc[i].to_dict()  # ligne → dictionnaire
        label = int(y.iloc[i])        # label réel
        yield x, label

# Démonstration sur 3 exemples
print('Les 3 premiers exemples du flux :')
for i, (x, label) in enumerate(simuler_flux(X, y, limite=3)):
    print(f'\nExemple {i+1} :')
    print(f'  Label   : {label} ({"Normal" if label==0 else "Attaque"})')
    print(f'  Nb features : {len(x)}')
    print(f'  Premières valeurs : { {k: round(v,3) for k,v in list(x.items())[:4]} }')

print('\n→ river reçoit ces dictionnaires un par un via predict_one() et learn_one()')

Les 3 premiers exemples du flux :

Exemple 1 :
  Label   : 0 (Normal)
  Nb features : 42
  Premières valeurs : {'dur': 0.0, 'proto': 0.9, 'service': 0.0, 'state': 0.667}

Exemple 2 :
  Label   : 0 (Normal)
  Nb features : 42
  Premières valeurs : {'dur': 0.0, 'proto': 0.9, 'service': 0.0, 'state': 0.667}

Exemple 3 :
  Label   : 0 (Normal)
  Nb features : 42
  Premières valeurs : {'dur': 0.0, 'proto': 0.9, 'service': 0.0, 'state': 0.667}

→ river reçoit ces dictionnaires un par un via predict_one() et learn_one()


In [6]:
# ══════════════════════════════════════════
# INITIALISATION DES 3 MODÈLES
# ══════════════════════════════════════════

modeles = {
    # Modèle 1 : Hoeffding Tree
    # max_depth=10   → arbre max 10 niveaux de profondeur
    # grace_period=200 → attend 200 exemples avant de faire un split
    'HoeffdingTree': tree.HoeffdingTreeClassifier(
        max_depth=10,
        grace_period=200
    ),

    # Modèle 2 : KNN
    # n_neighbors=5 → regarde les 5 voisins les plus proches
    'KNN': neighbors.KNNClassifier(
        n_neighbors=5
    ),

    # Modèle 3 : SGD (régression logistique incrémentale)
    # Pas de paramètre obligatoire, fonctionne bien par défaut
    'SGD': linear_model.LogisticRegression(),
}

print('3 modèles initialisés :')
for nom in modeles:
    print(f'  ✓ {nom}')

3 modèles initialisés :
  ✓ HoeffdingTree
  ✓ KNN
  ✓ SGD


In [7]:
# ══════════════════════════════════════════
# ENTRAÎNEMENT SUR 5000 EXEMPLES
# On mesure : précision, kappa, latence
# ══════════════════════════════════════════

N = 5000  # nombre d'exemples à traiter

# Dictionnaire pour stocker les résultats de chaque modèle
resultats = {}

for nom, modele in modeles.items():
    print(f'\nEntraînement de {nom}...')

    # Métriques river
    acc   = metrics.Accuracy()    # précision globale
    kappa = metrics.CohenKappa()  # qualité (corrigée du hasard)

    # Listes pour les graphiques
    historique_precision = []
    historique_kappa     = []
    historique_latence   = []

    # ── BOUCLE PRINCIPALE DU FLUX ──
    for i, (x, label) in enumerate(simuler_flux(X, y, limite=N)):

        # 1. Mesurer le temps de prédiction
        t0   = time.perf_counter()
        pred = modele.predict_one(x)       # PRÉDIRE
        lat  = (time.perf_counter() - t0) * 1000  # en ms

        # 2. Mettre à jour les métriques
        if pred is not None:
            acc.update(label, pred)
            kappa.update(label, pred)

        # 3. Apprendre de cet exemple
        modele.learn_one(x, label)         # APPRENDRE

        # 4. Sauvegarder pour les graphiques (tous les 50 exemples)
        historique_precision.append(round(acc.get(), 4))
        historique_kappa.append(round(kappa.get(), 4))
        historique_latence.append(round(lat, 4))

    # Sauvegarder les résultats finaux
    resultats[nom] = {
        'accuracy'    : round(acc.get(), 4),
        'kappa'       : round(kappa.get(), 4),
        'latence_moy' : round(float(np.mean(historique_latence)), 4),
        'latence_max' : round(float(np.max(historique_latence)), 4),
        'precs'       : historique_precision,
        'kappas'      : historique_kappa,
        'latences'    : historique_latence,
    }

    print(f'  Accuracy  : {acc.get():.4f} ({acc.get()*100:.2f}%)')
    print(f'  Kappa     : {kappa.get():.4f}')
    print(f'  Latence   : moy={np.mean(historique_latence):.3f}ms | max={np.max(historique_latence):.3f}ms')

print('\n✓ Entraînement terminé pour les 3 modèles !')


Entraînement de HoeffdingTree...
  Accuracy  : 0.9778 (97.78%)
  Kappa     : 0.8021
  Latence   : moy=0.248ms | max=39.539ms

Entraînement de KNN...
  Accuracy  : 0.9738 (97.38%)
  Kappa     : 0.7737
  Latence   : moy=12.480ms | max=181.982ms

Entraînement de SGD...
  Accuracy  : 0.9870 (98.70%)
  Kappa     : 0.8753
  Latence   : moy=0.040ms | max=27.846ms

✓ Entraînement terminé pour les 3 modèles !


In [8]:
# ══════════════════════════════════════════
# COURBES DE PRÉCISION COMPARÉES
# ══════════════════════════════════════════

couleurs = {
    'HoeffdingTree': '#378ADD',
    'KNN'          : '#1D9E75',
    'SGD'          : '#7F77DD',
}

fig = go.Figure()

for nom, data in resultats.items():
    fig.add_trace(go.Scatter(
        x    = list(range(len(data['precs']))),
        y    = data['precs'],
        name = f"{nom} (final: {data['accuracy']*100:.2f}%)",
        mode = 'lines',
        line = dict(color=couleurs[nom], width=2),
    ))

# Ligne de référence = prédiction aléatoire
fig.add_hline(
    y=0.5, line_dash='dash', line_color='gray', opacity=0.5,
    annotation_text='Baseline aléatoire (50%)',
)

fig.update_layout(
    title  = f'Courbes de précision préquentielle — 3 modèles ({N} exemples)',
    xaxis_title = 'Exemple #',
    yaxis_title = 'Précision',
    yaxis_range = [0.4, 1.05],
    height = 420,
    legend = dict(orientation='h', y=1.12),
)
fig.show()

print('Lecture du graphique :')
print('→ La courbe monte = le modèle apprend de plus en plus bien')
print('→ Plus la courbe est haute = meilleur modèle')
print('→ La vitesse de montée = vitesse d\'apprentissage')

Lecture du graphique :
→ La courbe monte = le modèle apprend de plus en plus bien
→ Plus la courbe est haute = meilleur modèle
→ La vitesse de montée = vitesse d'apprentissage


In [9]:
# ══════════════════════════════════════════
# COURBES DE KAPPA COMPARÉES
# Kappa = précision corrigée du hasard
# Plus fiable que l'accuracy sur les datasets déséquilibrés
# ══════════════════════════════════════════

fig2 = go.Figure()

for nom, data in resultats.items():
    fig2.add_trace(go.Scatter(
        x    = list(range(len(data['kappas']))),
        y    = data['kappas'],
        name = f"{nom} (Kappa final: {data['kappa']:.3f})",
        mode = 'lines',
        line = dict(color=couleurs[nom], width=2),
    ))

fig2.update_layout(
    title  = 'Courbes de Kappa — 3 modèles',
    xaxis_title = 'Exemple #',
    yaxis_title = 'Cohen Kappa',
    height = 380,
    legend = dict(orientation='h', y=1.12),
)
fig2.show()

print('Interprétation du Kappa :')
print('  0.00 - 0.20 → Très faible')
print('  0.20 - 0.40 → Faible')
print('  0.40 - 0.60 → Modéré')
print('  0.60 - 0.80 → Bien ✓')
print('  0.80 - 1.00 → Très bien ✓✓')

Interprétation du Kappa :
  0.00 - 0.20 → Très faible
  0.20 - 0.40 → Faible
  0.40 - 0.60 → Modéré
  0.60 - 0.80 → Bien ✓
  0.80 - 1.00 → Très bien ✓✓


In [10]:
# ══════════════════════════════════════════
# GRAPHIQUE : Comparaison des latences
# ══════════════════════════════════════════

noms    = list(resultats.keys())
lat_moy = [resultats[n]['latence_moy'] for n in noms]
lat_max = [resultats[n]['latence_max'] for n in noms]

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    name='Latence moyenne',
    x=noms, y=lat_moy,
    marker_color=[couleurs[n] for n in noms],
    text=[f'{v:.3f} ms' for v in lat_moy],
    textposition='outside',
))
fig3.add_trace(go.Bar(
    name='Latence max',
    x=noms, y=lat_max,
    marker_color=[couleurs[n] for n in noms],
    opacity=0.4,
    text=[f'{v:.3f} ms' for v in lat_max],
    textposition='outside',
))

fig3.update_layout(
    title   = 'Latence de prédiction par modèle (ms)',
    barmode = 'group',
    height  = 380,
    yaxis_title = 'Latence (ms)',
)
fig3.show()

print('Analyse :')
print(f'  SGD est le plus rapide  : {resultats["SGD"]["latence_moy"]:.3f} ms')
print(f'  KNN est le plus lent    : {resultats["KNN"]["latence_moy"]:.3f} ms')
print(f'  HoeffdingTree au milieu : {resultats["HoeffdingTree"]["latence_moy"]:.3f} ms')
print('→ Tous sont assez rapides pour un système temps réel (< 2ms)')

Analyse :
  SGD est le plus rapide  : 0.040 ms
  KNN est le plus lent    : 12.480 ms
  HoeffdingTree au milieu : 0.248 ms
→ Tous sont assez rapides pour un système temps réel (< 2ms)


In [11]:
# ══════════════════════════════════════════
# TABLEAU COMPARATIF
# ══════════════════════════════════════════

rows = []
for nom, data in resultats.items():
    rows.append({
        'Modèle'         : nom,
        'Accuracy'       : f"{data['accuracy']*100:.2f}%",
        'Kappa'          : f"{data['kappa']:.4f}",
        'Latence moy (ms)': f"{data['latence_moy']:.3f}",
        'Latence max (ms)': f"{data['latence_max']:.3f}",
        'Mémoire'        : 'Faible' if nom in ['HoeffdingTree','SGD'] else 'Élevée',
        'Vitesse'        : 'Très rapide' if nom == 'SGD' else ('Rapide' if nom == 'HoeffdingTree' else 'Lent'),
    })

df_compare = pd.DataFrame(rows)
print('Tableau comparatif des 3 modèles :')
print(df_compare.to_string(index=False))

print('\n' + '='*60)
print('CONCLUSION')
print('='*60)
print(f'  Meilleure accuracy : SGD ({resultats["SGD"]["accuracy"]*100:.2f}%)')
print(f'  Meilleur Kappa     : SGD ({resultats["SGD"]["kappa"]:.4f})')
print(f'  Plus rapide        : SGD ({resultats["SGD"]["latence_moy"]:.3f} ms)')
print(f'  Plus lent          : KNN ({resultats["KNN"]["latence_moy"]:.3f} ms)')
print()
print('→ Semaine 2 : le bandit ε-greedy choisira automatiquement')
print('  le meilleur modèle selon les données en temps réel.')
print('→ Même si SGD est meilleur maintenant, ça peut changer')
print('  si la nature du trafic réseau évolue (concept drift).')

Tableau comparatif des 3 modèles :
       Modèle Accuracy  Kappa Latence moy (ms) Latence max (ms) Mémoire     Vitesse
HoeffdingTree   97.78% 0.8021            0.248           39.539  Faible      Rapide
          KNN   97.38% 0.7737           12.480          181.982  Élevée        Lent
          SGD   98.70% 0.8753            0.040           27.846  Faible Très rapide

CONCLUSION
  Meilleure accuracy : SGD (98.70%)
  Meilleur Kappa     : SGD (0.8753)
  Plus rapide        : SGD (0.040 ms)
  Plus lent          : KNN (12.480 ms)

→ Semaine 2 : le bandit ε-greedy choisira automatiquement
  le meilleur modèle selon les données en temps réel.
→ Même si SGD est meilleur maintenant, ça peut changer
  si la nature du trafic réseau évolue (concept drift).


In [13]:
# ══════════════════════════════════════════
# BLOC FINAL : GRAPHIQUE RÉCAPITULATIF
# ══════════════════════════════════════════
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Sécurité pour l'affichage dans VS Code
pio.renderers.default = 'vscode'

# Configuration des sous-graphiques (1 ligne, 3 colonnes)
fig4 = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Accuracy (%)', 'Cohen Kappa', 'Latence moy (ms)']
)

# Extraction des données depuis votre dictionnaire 'resultats'
# On utilise .get() pour éviter les plantages si une donnée manque
acc_vals = [resultats[n].get('accuracy', 0) * 100 for n in noms]
kap_vals = [resultats[n].get('kappa', 0) for n in noms]
lat_vals = [resultats[n].get('latence_moy', 0) for n in noms]
cols_list = [couleurs.get(n, '#gray') for n in noms]

# Boucle pour créer les 3 barcharts
for i, (vals, fmt) in enumerate([
    (acc_vals, '.2f'),
    (kap_vals, '.4f'),
    (lat_vals, '.3f'),
]):
    fig4.add_trace(
        go.Bar(
            x=noms, 
            y=vals,
            marker_color=cols_list,
            text=[f'{v:{fmt}}' for v in vals],
            textposition='outside',
            showlegend=False,
        ),
        row=1, col=i+1
    )

# Mise en page
fig4.update_layout(
    title  = 'Comparaison finale — 3 modèles Online Learning (Flux)',
    height = 450,
    template = 'plotly_white'
)

fig4.show()

print('✓ Semaine 1 terminée !')
print('  Livrable : 3 modèles river fonctionnels + notebook EDA + notebook modèles')
print('-' * 50)
print('Prochaine étape (Semaine 2) :')
print('→ Implémentation du bandit ε-greedy pour l\'Auto-ML')
print('→ Ajout de la détection de dérive (Concept Drift) avec ADWIN')

✓ Semaine 1 terminée !
  Livrable : 3 modèles river fonctionnels + notebook EDA + notebook modèles
--------------------------------------------------
Prochaine étape (Semaine 2) :
→ Implémentation du bandit ε-greedy pour l'Auto-ML
→ Ajout de la détection de dérive (Concept Drift) avec ADWIN
